# Chapter 15 — Build It Yourself: RLHF & DPO

Go past *imitation* to *preference*. Watch an SFT model default to **curt** replies, train a **reward model** from comparisons, run the **RLHF** policy gradient, then implement **DPO** and warm the model up — all on a CPU.

> 💡 **No GPU needed.** Running a ✍️ cell before you fill it prints a friendly ⚠️, not an error.

### Setup — run this (the model, the reward model & the helpers; you don't edit it)

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F

# preference triples: (prompt, chosen = warm, rejected = curt), length-balanced so the preference is TONE
PREFS = [('hi','hello there','what do you want'), ('bye','take good care','finally you leave'),
         ('thanks','my pleasure','yeah sure fine'), ('hey','great to see you','oh no its you'),
         ('help','glad to help','figure it out')]
U, A, E = '\x01', '\x02', '\x03'                          # <user>, <assistant>, <end> markers
texts = [U+p+A+c+E for p,c,r in PREFS] + [U+p+A+r+E for p,c,r in PREFS]
chars = sorted(set(''.join(texts))); stoi={c:i for i,c in enumerate(chars)}; itos={i:c for c,i in stoi.items()}
V = len(chars); BLK = max(len(t) for t in texts) + 16

def make_example(prompt, resp):                           # (x,y) with the PROMPT masked to -100
    seq=U+prompt+A+resp+E; ids=[stoi[c] for c in seq]; a=seq.index(A)
    return ids[:-1], [(t if p>=a else -100) for p,t in enumerate(ids[1:])]

def seq_logprob(model, prompt, resp):                     # sum log p(response tokens | prompt), differentiable
    seq=U+prompt+A+resp+E; ids=[stoi[c] for c in seq]; y=torch.tensor(ids[1:])
    logp=torch.log_softmax(model(torch.tensor(ids[:-1]).unsqueeze(0))[0], -1)
    return logp[torch.arange(len(y)), y][seq.index(A):].sum()

class Blk(nn.Module):
    def __init__(s,C,H):
        super().__init__(); s.l1,s.l2=nn.LayerNorm(C),nn.LayerNorm(C); s.nh,s.hd=H,C//H
        s.q,s.k,s.v,s.pj=(nn.Linear(C,C) for _ in range(4)); s.ff=nn.Sequential(nn.Linear(C,4*C),nn.GELU(),nn.Linear(4*C,C))
    def forward(s,x):
        B,T,C=x.shape; q,k,v=(l(x).view(B,T,s.nh,s.hd).transpose(1,2) for l in (s.q,s.k,s.v))
        m=torch.triu(torch.ones(T,T)*float('-inf'),1)
        x=x+s.pj((torch.softmax((q@k.transpose(-2,-1))/math.sqrt(s.hd)+m,-1)@v).transpose(1,2).reshape(B,T,C))
        return x+s.ff(s.l2(x))

class Body(nn.Module):                                    # shared backbone: embeddings + blocks -> hidden state
    def __init__(s):
        super().__init__(); s.tok=nn.Embedding(V,64); s.pos=nn.Embedding(BLK,64)
        s.blocks=nn.ModuleList([Blk(64,4) for _ in range(2)]); s.lnf=nn.LayerNorm(64)
    def forward(s,idx):
        x=s.tok(idx)+s.pos(torch.arange(idx.size(1)))
        for b in s.blocks: x=b(x)
        return s.lnf(x)

class GPT(nn.Module):                                     # the policy: hidden -> next-token logits
    def __init__(s): super().__init__(); s.body=Body(); s.head=nn.Linear(64,V)
    def forward(s,idx): return s.head(s.body(idx))

class RewardModel(nn.Module):                             # same backbone, SCALAR head on the last token
    def __init__(s): super().__init__(); s.body=Body(); s.value=nn.Linear(64,1)
    def forward(s,idx): return s.value(s.body(idx))[:,-1,0]

def reward(rm, prompt, resp):                             # the reward model's score for one reply
    return rm(torch.tensor([stoi[c] for c in U+prompt+A+resp+E]).unsqueeze(0))[0]

@torch.no_grad()
def gen(model, prompt, greedy=True, max_new=18):          # generate model's reply (greedy or sampled)
    ids=[stoi[c] for c in U+prompt+A]
    for _ in range(max_new):
        lo=model(torch.tensor(ids).unsqueeze(0))[0,-1]
        nx=torch.argmax(lo).item() if greedy else torch.multinomial(torch.softmax(lo,-1),1).item()
        if itos[nx]==E: break
        ids.append(nx)
    return ''.join(itos[t] for t in ids[ids.index(stoi[A])+1:])

def train_reference(steps=400):                           # Chapter 14's SFT, weighted 2x toward the curt reply
    torch.manual_seed(0); m=GPT()
    ex=[make_example(p,c) for p,c,r in PREFS]+2*[make_example(p,r) for p,c,r in PREFS]
    opt=torch.optim.AdamW(m.parameters(), lr=3e-3)
    for _ in range(steps):
        loss=sum(F.cross_entropy(m(torch.tensor(x).unsqueeze(0)).view(-1,V),torch.tensor(y),ignore_index=-100) for x,y in ex)/len(ex)
        opt.zero_grad(); loss.backward(); opt.step()
    m.eval(); return m
print(f'{len(PREFS)} preference pairs | vocab {V} | block {BLK} — model, reward model & helpers ready')

## Step 1 — The curt reference  ▶️ Run this

Our starting point is a Chapter-14 SFT model trained (weighted toward the curt replies) to default to terse answers. That's our honest *before alignment* baseline — every method below will warm it up.

In [ ]:
ref = train_reference()
print('The SFT reference leans CURT — "before alignment" it greedily answers:')
for p,c,r in PREFS:
    print(f"  '{p}' -> {gen(ref,p)!r:20} (we'd prefer the warm {c!r})")

## Step 2 — The reward model: comparisons → a score  ✍️ Your turn

Humans give **comparisons** ("chosen is better than rejected"), not scores. The reward model learns a scalar from them with the **Bradley-Terry** loss: maximize `sigmoid(score(chosen) - score(rejected))`, i.e. minimize `-log sigmoid(score(chosen) - score(rejected))`.

<details><summary>Stuck? Show the answer</summary>

```python
return -F.logsigmoid(reward(rm, prompt, chosen) - reward(rm, prompt, rejected))
```
</details>

In [ ]:
# ✍️ Bradley-Terry: the reward model should score the CHOSEN reply above the REJECTED one.
#    loss = -log sigmoid( reward(chosen) - reward(rejected) )    [use reward(rm, prompt, resp) and F.logsigmoid]
def bt_loss(rm, prompt, chosen, rejected):
    return None      # replace

torch.manual_seed(1)
rm = RewardModel()
rm_trained = bt_loss(rm, *PREFS[0]) is not None
if rm_trained:
    opt = torch.optim.AdamW(rm.parameters(), lr=3e-3)
    for _ in range(300):
        loss = sum(bt_loss(rm, p, c, r) for p,c,r in PREFS) / len(PREFS)
        opt.zero_grad(); loss.backward(); opt.step()
    rm.eval()
    with torch.no_grad():
        for p,c,r in PREFS:
            print(f"  '{p}': chosen {reward(rm,p,c).item():+5.2f}  vs  rejected {reward(rm,p,r).item():+5.2f}")
else:
    print('bt_loss returns None — implement it above, then re-run.')

In [ ]:
# ▶️ Check your work
try:
    if not rm_trained:
        print('⚠️  Implement bt_loss above (return -log sigmoid of the score difference), then re-run.')
    else:
        with torch.no_grad():
            acc = sum(reward(rm,p,c) > reward(rm,p,r) for p,c,r in PREFS).item()
        if acc == len(PREFS):
            print(f'✅ The reward model ranks all {acc}/{len(PREFS)} pairs right (chosen scored above rejected).')
        else:
            print(f'⚠️  Only {acc}/{len(PREFS)} ranked right — the loss should be -log sigmoid(chosen - rejected).')
except Exception as e:
    print('❌', e)

## Step 3 — RLHF: optimize the policy with RL  ▶️ Run this

The classic pipeline. The policy **samples** replies, the reward model **scores** them, and **REINFORCE** pushes the good ones' probability up — with a **baseline** (reinforce reward *above average*) to cut variance and a **KL penalty** to keep the policy near the reference. (Uses the reward model you trained in Step 2. Takes ~10s.)

In [ ]:
# ▶️ RLHF: sample replies, score them with YOUR reward model, reinforce the good ones (+ a KL leash to ref).
if not rm_trained:
    print('⚠️  Train the reward model in Step 2 first — RLHF needs a reward model to optimize.')
else:
    torch.manual_seed(3)
    policy = GPT(); policy.load_state_dict(ref.state_dict())
    before = {p: gen(policy,p) for p,c,r in PREFS}
    opt = torch.optim.AdamW(policy.parameters(), lr=3e-4); baseline = 0.0; KL_COEF = 0.3
    for step in range(100):
        loss, rews = 0.0, []
        for p,c,r in PREFS:
            for _ in range(3):
                with torch.no_grad():
                    resp = gen(policy, p, greedy=False)
                    rew = reward(rm, p, resp).item(); ref_lp = seq_logprob(ref, p, resp)
                rews.append(rew); lp = seq_logprob(policy, p, resp)
                loss = loss - (rew - baseline)*lp + KL_COEF*(lp - ref_lp)     # REINFORCE + KL
        loss = loss/(len(PREFS)*3); opt.zero_grad(); loss.backward(); opt.step()
        baseline = 0.9*baseline + 0.1*(sum(rews)/len(rews))
        if step % 25 == 0 or step == 99: print(f'  step {step:2d}: mean reward {sum(rews)/len(rews):+.2f}')
    policy.eval()
    print('\nRLHF before -> after (the reward model pulled the policy warm):')
    for p,c,r in PREFS: print(f"  '{p}': {before[p]!r:18} -> {gen(policy,p)!r:18} (chosen {c!r})")

## Step 4 — DPO: alignment without a reward model  ✍️ Your turn — the star

DPO skips the reward model *and* the RL loop. Each reply's **implicit reward** is `β·(logp_policy − logp_ref)`; plug it into the same Bradley-Terry loss and train it like plain supervised learning. Implement the loss and watch curt → warm.

<details><summary>Stuck? Show the answer</summary>

```python
return -F.logsigmoid(BETA * ((pc - rc) - (pr - rr)))
```
</details>

In [ ]:
# ✍️ DPO: each reply's IMPLICIT reward is  r_hat(y) = BETA*(logp_policy(y) - logp_ref(y)).
#    Then Bradley-Terry on chosen vs rejected:  loss = -log sigmoid( r_hat(chosen) - r_hat(rejected) )
BETA = 0.3
def dpo_loss(policy, ref, prompt, chosen, rejected):
    pc, pr = seq_logprob(policy, prompt, chosen), seq_logprob(policy, prompt, rejected)
    with torch.no_grad():                                  # reference is frozen
        rc, rr = seq_logprob(ref, prompt, chosen), seq_logprob(ref, prompt, rejected)
    return None      # replace  (combine pc, pr, rc, rr and BETA into the DPO loss)

torch.manual_seed(2)
pol = GPT(); pol.load_state_dict(ref.state_dict())
for p in ref.parameters(): p.requires_grad = False        # freeze the reference
dpo_before = {p: gen(pol,p) for p,c,r in PREFS}
dpo_ok = dpo_loss(pol, ref, *PREFS[0]) is not None
if dpo_ok:
    opt = torch.optim.AdamW(pol.parameters(), lr=2e-4)
    for _ in range(100):
        loss = sum(dpo_loss(pol, ref, p, c, r) for p,c,r in PREFS) / len(PREFS)
        opt.zero_grad(); loss.backward(); opt.step()
    pol.eval()
    print('DPO before -> after (no reward model, no RL loop):')
    for p,c,r in PREFS: print(f"  '{p}': {dpo_before[p]!r:18} -> {gen(pol,p)!r:18} (chosen {c!r})")
else:
    print('dpo_loss returns None — implement it above, then re-run.')

In [ ]:
# ▶️ Check your work
try:
    if not dpo_ok:
        print('⚠️  Implement dpo_loss above (return -log sigmoid of the implicit-reward difference), then re-run.')
    else:
        with torch.no_grad():
            acc = sum(seq_logprob(pol,p,c) > seq_logprob(pol,p,r) for p,c,r in PREFS).item()
        if acc == len(PREFS):
            print(f'✅ DPO aligned the policy: it now prefers the warm reply on {acc}/{len(PREFS)} prompts — no reward model, no RL.')
        else:
            print(f'⚠️  {acc}/{len(PREFS)} preferred — check the sign: -log sigmoid(r_hat(chosen) - r_hat(rejected)).')
except Exception as e:
    print('❌', e)

## 🎉 You built alignment.

You trained a reward model from comparisons, ran the RLHF policy gradient, and implemented DPO — two routes to the same place: a model that prefers the replies people prefer. This is the last training stage of a chat model. Take **DPO** to your own model in the [mini-project](../project/), then on to [Chapter 16 — Deployment](../../16-deployment/). 🎓